# LÀM QUEN VỚI DỮ LIỆU ViHOS

# 1. Dataset dùng để làm gì 

Hãy tưởng tượng VIHOS có rất nhiều bài như thế này: Câu "Mày đúng là đồ ngốc" và đáp án "đồ ngốc". Model của ta sẽ nhìn rất nhiều ví dụ câu + đáp án đúng, từ đó học cách tự tìm đáp án khi gặp câu mới. 

ViHOS có tổng cộng 11.056 bình luận, trong đó:
- 5.360 bình luận có đoạn hate/offensive
- 5.696 bình luận không có đoạn nào
- Tổng cộng có 26.476 đoạn được con người đánh dấu

# 2. Dataset được chia thành 3 phần

| Tập | Số bình luận | Hiểu đơn giản |
|---|---:|---|
| **Train** | 8.844 | Bộ bài cho model **học** |
| **Dev** | 1.106 | Bộ bài **thi thử** trong lúc ta phát triển model |
| **Test** | 1.106 | Bộ bài **thi thật** cuối cùng | ([GitHub][1])

[1]: https://github.com/phusroyal/ViHOS "GitHub - phusroyal/ViHOS: Repository for the paper \"ViHOS: Vietnamese Hate and Offensive Spans Detection\" (EACL2023) · GitHub"

# 3. ViHOS cho dữ liệu theo 2 cách

Trong repository chính thức, có cấu trúc gần như thế này:
- data/Sequence labeling-based version
    - data/Sequence labeling-based version/Syllable/
    - data/Sequence labeling-based version/Word/
- data/Span Extraction-based version/
- data/Test_data/

Giả sử ta có một câu: "Mày đúng là đồ ngốc thật" và đáp án đúng là "đồ ngốc". Bản chất bài toán vẫn chỉ có một đáp án duy nhất: Trong câu này, đoạn xúc phạm là "đồ ngốc". Nhưng máy tính có nhiều cách để ghi lại đáp án đó

ViHOS cung cấp hai cách chính:
- Cách A - Ghi vị trí đoạn cần tìm
    - Kiểu như: "Đoạn xúc phạm bắt đầu ở ký tự số X và kết thúc ở ký tự số Y"
    - Đây là Span Detection
- Cách B - Chia câu thành từng thành phần rồi đánh dấu từng phần

| Phần | Có thuộc đoạn xúc phạm không? |
|---|---|
| Mày | Không |
| đúng | Không |
| là | Không |
| đồ | Có, đây là chỗ bắt đầu |
| ngốc | Có, tiếp tục |
| thật | Không |

- Đây là Sequence Labeling


**Cách 1 - Lưu vị trí của đoạn cần tìm**

Ta lấy một ví dụ đơn giản hơn: "Anh ta rất ngu", con người nhìn vào là biết đoạn cần tìm = "ngu". Nhưng máy tính không nhất thiết phải lưu chữ "ngu" làm đáp án, nó có thể lưu vị trí của chữ đó trong câu

Hãy đánh số các ký tự, index: Câu "Anh ta rất ngu" có Span: [11, 14]. Nghĩa là hãy lấy đoạn từ vị trí này đến vị trí kia, và khi lấy ra ta được "ngu"

Nếu câu có 2 đoạn xấu thì span = [vị trí span 1, vị trí span 2]

Vậy trong kiểu dữ liệu này ta có hai thứ quan trọng:
- `content`: Câu gốc
- `index_spans`: Danh sách vị trí của những đoạn cần highlight

**Cách 2 - Sequence Labeling**

Bước 1: Chia câu thành nhiều phần
- Ví dụ: Mày, đúng, là, đồ, ngốc, thật

Bước 2: Xét từng phần
- Ta biết đáp án là đồ ngốc, vậy: Mày -> không, đúng -> không, là -> không, đồ -> không, ngốc -> không, thật -> không, đồ -> có, ngốc -> có, thật -> không
- Nếu chỉ dùng: Có/Không thì vẫn còn một vấn đề. Giả sử: "đồ ngu dốt" thì ta biết ba phần đều thuộc span, nhưng máy cần biết Span bắt đầu ở đâu, vì vậy người ta dùng ba nhãn: O, B-T, I-T
    - O: Không thuộc đoạn cần tìm
    - B-T: Đây là phần đầu tiên của một span, ví dụ Đồ ngu dốt thì Đồ -> B-T
    - I-T: Đoạn đó vẫn đang tiếp tục, ví dụ đồ -> B-T, ngu -> I-T, dốt -> I-T
    - Ghép lại toàn bộ câu:

| Phần | Nhãn |
|---|---|
| Mày | O |
| đúng | O |
| là | O |
| đồ | B-T |
| ngu | I-T |
| dốt | I-T |
| thật | O |

- Vậy model chỉ cần học: Với mỗi phần của câu, nên gắn O, B-T hay I-T

Tại sao ta chọn Sequence Labeling:
- Chưa cần đi vào model, chỉ cần hiểu lý do cơ bản: Dataset đã cung cấp sẵn dạng: phần của câu + nhãn nên ta có thể dạy model: "Nhìn từng phần và dự đoán xem nó thuộc loại nào"

**Tại sao ViHOS lại cung cấp cả Syllable (tiếng) và Word (từ)**

Ta có thể hỏi: Model làm tốt hơn khi nhìn từng tiếng hay khi nhìn từng từ. Ví dụ:
- Model A nhìn Syllable: học | sinh
- Model B nhìn Word: học_sinh

Model B có thể hiểu tốt hơn rằng học_sinh là một đơn vị hoàn chỉnh. Nhưng không có nghĩa Word luôn luôn tốt hơn, vì tách từ cũng có thể sai, nhất là dữ liệu mạng xã hội có: viết tắt, teencode, sai chính tả, từ lóng, cách viết rất không chuẩn. Đó là lý do việc so sánh hai cách này có ý nghĩa cho đồ án. ViHOS cũng xem dữ liệu ở mức syllable trong các thống kê chính của dataset